# Workshop — 1. Generate Imitation-Learning Data

In this notebook, we build the dataset that will be used to adapt a **Vision-Language-Action model**.

The complete workshop flow is:

```text
Notebook 1                    Notebook 2                  Notebook 3
scripted teacher ─ dataset ─▶ zero-shot SmolVLA fails ─▶ fine-tuning and comparison
```

The first two notebooks intentionally use the same contract:

- robot: **SO100** in MuJoCo simulation;
- task: `Pick up the cube and place it in the box.`;
- cameras: `top` and `wrist`;
- state and action: six SO100 joints;
- metric: `placed_in_box`.

> The teacher reads the cube's ground-truth position from MuJoCo. This is a useful privilege for generating demonstrations, not a capability we attribute to the VLA.

## 1. Installation

We use the same runtime revision as Notebook 2. The critical path remains simulation-only: no hardware is initialized.

In [ ]:
%pip install -q -r requirements.txt

## 2. Environment and Imports

`MUJOCO_GL=cgl` enables off-screen rendering on macOS. It must be set before importing MuJoCo.

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

In [ ]:
import json
import sys
from pathlib import Path

from IPython.display import Video, display
from PIL import Image

sys.path.insert(0, str(Path("code").resolve()))

from scenarios import get_scenario, list_scenarios

SCENARIO_NAME = "so100_pick_place"
scenario = get_scenario(SCENARIO_NAME)

print("Available scenarios:", list_scenarios())
print("Selected scenario:", scenario.name)
print("Task:", scenario.instruction)
print("Action keys:", scenario.joint_keys)

## 3. Select a Registered Scenario

Each simulation task lives in its own file under `code/scenarios/` and exposes one explicit `SimulationScenario` contract: scene construction, instruction, robot and joint keys, cameras, episode randomization, teacher factory, dataset location, and success metric.

Set `SCENARIO_NAME` above to a registered name. Registration is explicit in `code/scenarios/__init__.py`; filenames are not discovered through a hidden naming convention. Notebook 2 and Notebook 4 use the same default scenario through `vla_pick.py`.

In [ ]:
sim = scenario.build_scene()
observation = sim.get_observation(scenario.robot_name)

for camera_name in scenario.camera_names:
    print(camera_name)
    display(Image.fromarray(observation[camera_name]).resize((480, 360)))
print(scenario.diagnostics(sim))

Each dataset frame for the selected scenario will contain:

```text
observation.images.<scenario camera> ─┐
observation.state ────────────────────┼─▶ state + instruction ─▶ robot action
task string ──────────────────────────┘
```

The scripted policy does not use the images: it computes a privileged trajectory from simulator state. The images are still recorded because they will be the VLA inputs. The current `so100_pick_place` scenario produces the concrete `top` and `wrist` streams used by the remaining notebooks.

## 4. The Scripted Teacher

The teacher executes ten phases: open, approach, descend, close, lift, carry, lower into the box, release, retreat, and settle.

For each cube position, it corrects the waypoints with inverse kinematics on the actual jaw center. Actions between waypoints use cosine interpolation, avoiding discontinuities in both the servos and the dataset.

In [ ]:
teacher = scenario.make_teacher(sim)
print("Policy:", type(teacher).__name__)
print("Provider:", teacher.provider_name)
print("Uses images:", teacher.requires_images)
print("Steps:", teacher.n_steps)
print("Phases:", teacher.phase_boundaries)

### One Demonstration Before Collection

`status="success"` only means that the rollout executed. We separately verify that the cube is actually inside the target box.

In [ ]:
TEACHER_VIDEO = Path("so100_teacher.mp4").resolve()

result = sim.run_policy(
    robot_name=scenario.robot_name,
    policy_object=teacher,
    instruction=scenario.instruction,
    n_steps=teacher.n_steps,
    control_frequency=scenario.fps,
    fast_mode=True,
    video={
        "path": str(TEACHER_VIDEO),
        "camera": scenario.video_camera,
        "fps": scenario.fps,
    },
)
diagnostics = scenario.diagnostics(sim)
print("run_policy status:", result["status"])
print("task diagnostics:", diagnostics)

if result["status"] != "success" or not scenario.is_success(sim):
    raise RuntimeError("The demonstration failed: do not record this dataset.")

In [ ]:
display(Video(str(TEACHER_VIDEO), embed=True, width=640))

## 5. Data Collection

The recorder produces a `LeRobotDataset`: Parquet for state/action, MP4 for both cameras, and metadata under `meta/`.

Eight episodes are enough to complete the workshop quickly and verify the pipeline. For useful fine-tuning, increase `N_EPISODES` to at least 50 and carefully expand the scene variation.

In [ ]:
DATASET_ROOT = scenario.dataset_path(Path.cwd())
DATASET_REPO_ID = scenario.dataset_repo_id
DATASET_FPS = scenario.fps
N_EPISODES = 8

print(f"Dataset: {DATASET_ROOT}")
print(f"Episodes: {N_EPISODES}")
print("WARNING: overwrite=True sostituirà un dataset esistente in questo percorso.")

In [ ]:
def require_success(result, operation):
    if result.get("status") == "success":
        return result
    text = " | ".join(
        str(item.get("text"))
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )
    raise RuntimeError(f"{operation} failed: {text or result}")


require_success(
    sim.start_recording(
        repo_id=DATASET_REPO_ID,
        root=str(DATASET_ROOT),
        task=scenario.instruction,
        fps=DATASET_FPS,
        cameras=list(scenario.camera_names),
        overwrite=True,
    ),
    "start_recording",
)

episode_results = []
try:
    for episode in range(N_EPISODES):
        episode_metadata = scenario.prepare_episode(sim, episode)
        teacher = scenario.make_teacher(sim)

        rollout = sim.run_policy(
            robot_name=scenario.robot_name,
            policy_object=teacher,
            instruction=scenario.instruction,
            n_steps=teacher.n_steps,
            control_frequency=DATASET_FPS,
            fast_mode=True,
        )
        require_success(rollout, f"rollout episode {episode}")

        diagnostics = scenario.diagnostics(sim)
        success = scenario.is_success(sim)
        episode_results.append({
            "episode": episode,
            "episode_metadata": episode_metadata,
            "diagnostics": diagnostics,
            "success": success,
        })

        # Without this boundary, every rollout would become one long episode.
        require_success(sim.save_episode(), f"save episode {episode}")
        print(
            f"episode {episode:02d} | metadata={episode_metadata} | success={success}"
        )
finally:
    stop = sim.stop_recording()

require_success(stop, "stop_recording")
failed = [row for row in episode_results if not row["success"]]
if failed:
    raise RuntimeError(
        f"{len(failed)} failed demonstrations are present in the dataset: {failed}. "
        "Do not fine-tune on it; fix the cause and rerun with overwrite=True."
    )

print(f"Successful demonstrations: {len(episode_results)}/{N_EPISODES}")

## 6. Dataset Verification

We verify the episode count and the contract required for fine-tuning. The expected result is: 6D state, 6D action, a `top` camera, and a `wrist` camera.

In [ ]:
verification = sim.verify_dataset_episodes(expected=N_EPISODES)
require_success(verification, "verify_dataset_episodes")
print(verification["content"][0]["text"])

info = json.loads((DATASET_ROOT / "meta" / "info.json").read_text())
features = info["features"]

expected_images = scenario.image_feature_keys
actual_images = {name for name in features if name.startswith("observation.images.")}

assert info["total_episodes"] == N_EPISODES
assert info["robot_type"] == scenario.robot_name
assert info["fps"] == scenario.fps
assert features["observation.state"]["shape"] == [len(scenario.joint_keys)]
assert features["action"]["shape"] == [len(scenario.joint_keys)]
assert actual_images == expected_images

print("\nDataset summary")
print("  robot:   ", info.get("robot_type"))
print("  episodes:", info["total_episodes"])
print("  frames:  ", info["total_frames"])
print("  fps:     ", info["fps"])
print("  state:   ", features["observation.state"]["shape"])
print("  action:  ", features["action"]["shape"])
print("  cameras: ", sorted(actual_images))

In [ ]:
for camera_name in scenario.camera_names:
    videos = sorted(
        DATASET_ROOT.glob(f"videos/observation.images.{camera_name}/**/*.mp4")
    )
    if not videos:
        raise RuntimeError(f"Missing dataset video stream for {camera_name!r}.")
    print(camera_name)
    display(Video(str(videos[0]), embed=True, width=480))

## Result and Next Step

We now have **SO100 demonstrations compatible with the task in Notebook 2**:

- the same `top` + `wrist` images;
- the same instruction;
- the same 6D state and 6D action;
- success verified with the same `placed_in_box` metric.

In Notebook 2, we will instead run a checkpoint trained on real-world images: inference will work, but the robot will probably fail because of domain shift.

Notebook 3 will use this dataset for fine-tuning and load the new checkpoint with **radian** units, matching the values recorded by the simulator.